In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import CrossEntropyLoss
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from bs4 import BeautifulSoup
import re

In [3]:
train = pd.read_csv(
    "../data/labeledTrainData.tsv",
    header=0,
    delimiter="\t",
    quoting=3
)

In [7]:
def clean_review_for_roberta(review):
    # 去 HTML
    review_text = BeautifulSoup(review, "html.parser").get_text()

    # RoBERTa 是 BPE 模型，不需要强行小写或去标点
    review_text = re.sub(r"[^A-Za-z0-9!?',.\" ]", " ", review_text)

    # 去掉多余空格
    review_text = re.sub(r"\s+", " ", review_text).strip()

    return review_text

clean_train_reviews = [clean_review_for_roberta(r) for r in train["review"]]

In [4]:
tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 9e274062-ea04-42ea-b951-956392429f50)')' thrown while requesting HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 121dcdb9-3376-47fe-915f-ee40455185c9)')' thrown while requesting HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 27253b37-d062-404c-89f1-98ea55361d69)')' thrown while requesting HEAD https://huggingface.co/roberta-base/resolve/main/tokenizer_config.json
Retrying in 4s [Retry 3/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Reque

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

D:\workspace\Python\miniconda3\envs\DeepLearning_py310\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\TANG\.cache\huggingface\hub\models--roberta-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

In [5]:
class IMDBDataset(Dataset):
    def __init__(self, reviews, labels=None, max_len=256):
        self.reviews = reviews
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.reviews)

    def __getitem__(self, idx):
        review = str(self.reviews[idx])
        encoding = tokenizer.encode_plus(
            review,
            add_special_tokens=True,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt"
        )
        item = {
            "input_ids": encoding["input_ids"].flatten(),
            "attention_mask": encoding["attention_mask"].flatten()
        }
        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

In [8]:
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

num_epochs = 3
batch_size = 16
learning_rate = 2e-5

train_labels = torch.tensor(train['sentiment'].values, dtype=torch.long)
train_dataset = IMDBDataset(clean_train_reviews, train_labels)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)


optimizer = AdamW(model.parameters(), lr=learning_rate)
total_steps = len(train_loader) * num_epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=0,
    num_training_steps=total_steps
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

loss_fn = CrossEntropyLoss()

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ea136577-2820-4353-9848-1ca6300277b7)')' thrown while requesting HEAD https://huggingface.co/roberta-base/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: e1cd878d-75b5-4146-b373-9ca4acea81f3)')' thrown while requesting HEAD https://huggingface.co/roberta-base/resolve/main/config.json
Retrying in 2s [Retry 2/5].
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        total_loss += loss.item()

        loss.backward()
        optimizer.step()
        scheduler.step()

    print(f"Epoch {epoch+1}, Loss: {total_loss / len(train_loader):.4f}")

C:\Users\TANG\AppData\Local\Temp\ipykernel_17828\2831193578.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)


Epoch 1, Loss: 0.2260
Epoch 2, Loss: 0.1246
Epoch 3, Loss: 0.0585


In [10]:
test = pd.read_csv(
    "../data/testData.tsv",
    header=0,
    delimiter="\t",
    quoting=3
)

clean_test_reviews = [clean_review_for_roberta(r) for r in test["review"]]

test_dataset = IMDBDataset(clean_test_reviews)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [11]:
model.eval()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        preds = torch.argmax(outputs.logits, dim=1)
        predictions.extend(preds.cpu().numpy())

In [12]:
output = pd.DataFrame({
    "id": test["id"],
    "sentiment": predictions
})
output.to_csv("RoBERTa_IMDB_predictions.csv", index=False, quoting=3)

print("Done! File saved: RoBERTa_IMDB_predictions.csv")

Done! File saved: RoBERTa_IMDB_predictions.csv
